In [1]:
from pyspark.sql.functions import *

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 3, Finished, Available, Finished, False)

In [2]:
employees_df = spark.read.table("LH_Nanda.InnovateSolutions.Bronze_Employees")
display(employees_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a84569d7-31bc-4b29-80bf-7fc46754dddb)

In [3]:
departments_df = spark.read.csv("Files/CaseStudy4/departments.csv",header=True,inferSchema=True)

display(departments_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 85ede61b-5273-4b26-97ce-e84a3df8cbb4)

In [4]:
null_df = employees_df.filter(col("hire_date").isNull()  | col("salary").isNull()).withColumn("quarantine_reason",lit("NULL_FIELD"))
display(null_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe89120f-1dc5-462c-a186-71f356d90231)

In [5]:
invalid_df = employees_df.filter( (col("salary")<0)  | (col("salary")==9999999) ).withColumn("quarantine_reason",lit("INVALID_SALARY"))
display(invalid_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f85172ec-b913-414d-b2b1-023d78f9a0d8)

In [6]:
quarantine_df=null_df.unionByName(invalid_df)
display(quarantine_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd7576a7-0e3b-4a16-ba81-952774227b78)

In [7]:
working_df = employees_df.join(quarantine_df,"employee_id","left_anti")
display(working_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4892db0b-5055-41d2-abed-c1592f1377d0)

In [8]:
dedup_df = working_df.orderBy(col("updated_at").desc()).dropDuplicates(["employee_id"])
display(dedup_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 99e954b8-ac60-4876-96ef-84e7901eef81)

In [23]:
enriched_df = dedup_df.withColumn("hire_year",year("hire_date")).withColumn("Tenure",round(floor(months_between(current_date(),col("hire_date")))/12,2))
display(enriched_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4578d052-393e-4ee3-b6ef-937bb144bdfe)

In [43]:
def get_tenure_group(x):
    return when(x>=5,"Veteran") .when((x>=2) & (x<=5),"Established") .when(x<2,"New").otherwise("unknown")

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 45, Finished, Available, Finished, False)

In [49]:
enriched_df = dedup_df.withColumn("hire_year",year("hire_date"))\
            .withColumn("tenure",round(floor(months_between(current_date(),col("hire_date")))/12,2))\
            .withColumn("salary_brand",when(col("salary")>=120000,"Senior")
            .when((col("salary")>=80000) & ( col("salary")<=119999),"Mid")
            .when(col("salary")<80000,"Junior").otherwise("unknown"))\
            .withColumn("tenure_group",get_tenure_group(col("tenure")))\
            .join(departments_df,"department_id","left").withColumn("transformed_at",current_timestamp())

            
display(enriched_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78ee0f4a-48e9-4899-8407-3472bf23ab4b)

In [48]:
joined_df = enriched_df. # added in above row
display(joined_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e6892c79-58c6-4964-b4b1-b2a1e67fc88c)

In [54]:
protected_df = enriched_df.withColumn("employee_name_bash",sha2(col("employee_name"),256))\
                .withColumn("salary_masked",expr("repeat('*',length(salary))"))\
                .drop("employee_name","salary")
display(protected_df)

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 407b7a8a-2a29-4a15-bdfc-390ff75a1896)

In [56]:
protected_df.write.mode("overwrite").saveAsTable("LH_Nanda.InnovateSolutions.Silver_Employees")
quarantine_df.write.mode("overwrite").saveAsTable("LH_Nanda.InnovateSolutions.Silver_Quarantine")

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 58, Finished, Available, Finished, False)

In [57]:
print(f"Bronze:{employees_df.count()}")
print(f"Silver_accepted:{protected_df.count()}")
print(f"silver_duplicates:{dedup_df.count()}")
print(f"silver_quarantine:{quarantine_df.count()}")

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 59, Finished, Available, Finished, False)

Bronze:332
Silver_accepted:307
silver_duplicates:307
silver_quarantine:20


In [63]:
print(f"Bronze:{employees_df.count()}")
print(f"Silver_accepted:{protected_df.count()}")
print(f"silver_duplicates:{working_df.count()-dedup_df.count()}")
print(f"silver_quarantine:{quarantine_df.count()}")
print(f"CheckSum:{employees_df.count() - protected_df.count() - working_df.count() + dedup_df.count() - quarantine_df.count()}")

StatementMeta(, 0936bcfa-0f6b-4605-afa5-a89bd4d3a982, 65, Finished, Available, Finished, False)

Bronze:332
Silver_accepted:307
silver_duplicates:5
silver_quarantine:20
CheckSum:0
